# Attention-Free Geometric Reasoning: Locked Four-Seed Study

## Required download and upload

Before running this notebook:

1. Open the public [`ERGT-paper` repository](https://github.com/jalaljafari2009/ERGT-paper), choose **Code > Download ZIP**, and extract the downloaded `ERGT-paper-main.zip`.
2. In Colab, select a GPU runtime and choose **Run all**.
3. When the first code cell opens the file picker, upload exactly the inner file `ERGT-paper-main/ERGT_FourSeed_Reproduction.zip` from the extracted folder. Do **not** upload the outer `ERGT-paper-main.zip`.

This notebook has one immutable scientific path; run all cells without editing. No historical Colab runtime version is required. It accepts Python 3.10+, PyTorch 2.0+, and current compatible NumPy/Pandas versions, and installs a required package only if it is missing or too old. Fresh outputs are written to Google Drive.

In [ ]:
from pathlib import Path
import os
import sys
import zipfile

os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
PACKAGE_NAME = "ERGT_FourSeed_Reproduction"
PACKAGE_ARCHIVE = f"{PACKAGE_NAME}.zip"
PACKAGE_PATH_HINT = f"ERGT-paper-main/{PACKAGE_ARCHIVE}"


def safe_extract(archive_path, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if destination != target and destination not in target.parents:
                raise RuntimeError(f"unsafe ZIP member: {member.filename}")
        archive.extractall(destination)


def locate_package():
    def complete_package_root(candidate):
        return (
            (candidate / "MANIFEST.sha256").is_file()
            and (candidate / "ergt_four_seed" / "__init__.py").is_file()
        )

    candidates = [Path.cwd(), Path.cwd() / PACKAGE_NAME, Path("/content") / PACKAGE_NAME]
    for candidate in candidates:
        if complete_package_root(candidate):
            return candidate.resolve()
    archives = [Path("/content") / PACKAGE_ARCHIVE, Path.cwd() / PACKAGE_ARCHIVE]
    archive = next((path for path in archives if path.is_file()), None)
    if archive is None:
        from google.colab import files
        print(f"Required file: {PACKAGE_ARCHIVE}")
        print("Download the complete repository with GitHub Code > Download ZIP, then extract it.")
        print(f"Path inside the extracted GitHub download: {PACKAGE_PATH_HINT}")
        print("Upload the inner runtime package above; do not upload ERGT-paper-main.zip.")
        uploaded = files.upload()
        matches = [name for name in uploaded if Path(name).name == PACKAGE_ARCHIVE]
        if not matches:
            received = ", ".join(sorted(uploaded)) or "no file"
            raise RuntimeError(
                f"Upload {PACKAGE_ARCHIVE} from {PACKAGE_PATH_HINT}; received: {received}"
            )
        archive = Path(matches[0])
    safe_extract(archive, "/content")
    candidate = Path("/content") / PACKAGE_NAME
    if not complete_package_root(candidate):
        raise RuntimeError(
            f"{PACKAGE_ARCHIVE} did not contain the complete standalone package"
        )
    return candidate.resolve()


PACKAGE_ROOT = locate_package()
sys.path.insert(0, str(PACKAGE_ROOT))
print("standalone_package_root=", PACKAGE_ROOT)
print("git_or_project_checkout_required=", False)


In [ ]:
from ergt_four_seed.environment import (
    ensure_required_packages,
    verify_environment,
)
from ergt_four_seed.integrity import verify_integrity
from ergt_four_seed.data_registry import verify_registered_data

integrity = verify_integrity(PACKAGE_ROOT)
dependency_setup = ensure_required_packages(install_missing=True)
import torch
environment = verify_environment(strict=True)
if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime before running this notebook")
data_parity = verify_registered_data()
print("integrity=", integrity)
print("dependency_setup=", dependency_setup)
print("environment=", environment)
print("registered_data_parity=", data_parity)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from ergt_four_seed import run_four_seed_study

OUTPUT_ROOT = Path("/content/drive/MyDrive/ERGT_FourSeed_Reproduction/runs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
result = run_four_seed_study(
    device="cuda",
    output_root=OUTPUT_ROOT,
    run_id="ergt_four_seed_confirmation",
    resume=True,
    copy_outputs_to_downloads=True,
    strict_environment=True,
)
print("summary_path=", result["summary_path"])
print("paper_report_path=", result["paper_report_path"])
print("stability_invariants_path=", result["stability_invariants_path"])
print("bundle_path=", result["bundle_path"])
print("execution_complete=", True)


In [ ]:
import json
import pandas as pd
from IPython.display import Markdown, display

root = Path(result["run_root"])
verdict = json.loads((root / "final_verdict.json").read_text(encoding="utf-8"))
table_root = root / "compact_tables"
table_manifest = json.loads((table_root / "compact_table_manifest.json").read_text(encoding="utf-8"))


def compact_table(filename):
    return pd.read_csv(table_root / filename)


display(Markdown("## Registered numerical readout"))
display(pd.DataFrame((
    {"quantity": "ERGT 32-hop accuracy", "value": verdict["native_32_hop_accuracy_mean"]},
    {"quantity": "Direct Transformer 32-hop accuracy", "value": verdict["direct_transformer_32_hop_accuracy_mean"]},
    {"quantity": "ERGT minus Direct Transformer at 32 hops", "value": verdict["native_minus_transformer_32_hop"]},
    {"quantity": "ERGT claim status", "value": verdict["native_ergt_claim_status"]},
    {"quantity": "Direct Transformer convergence readout", "value": verdict["direct_transformer_convergence_readout"]},
)))

display(Markdown("## Independent convergence by seed"))
display(compact_table("T03_independent_convergence.csv"))

mechanisms = compact_table("T07_registered_mechanism_matrix.csv")
heldout = mechanisms[mechanisms["panel"].eq("heldout_final")].copy()
display(Markdown("## Same-checkpoint geometric mechanism interventions"))
display(heldout.groupby(["scenario", "intervention"], as_index=False).agg(
    mean_full_accuracy=("full_accuracy", "mean"),
    mean_intervention_accuracy=("intervention_accuracy", "mean"),
    mean_targeted_drop=("targeted_drop", "mean"),
    evaluable_rows=("attribution_evaluable", "sum"),
))

geometry_controls = compact_table("T08_geometry_and_world_controls.csv")
expected_controls = {
    "shuffled_geometry", "random_geometry", "no_phi", "no_event_backreaction",
    "direct_world_gate", "no_multiscale_backbone", "no_world_transitions",
    *(f"only_world_{index}" for index in range(8)),
}
missing = expected_controls - set(geometry_controls["intervention"])
if missing:
    raise RuntimeError(f"missing registered geometry controls: {sorted(missing)}")
display(Markdown("## Randomized, shuffled, ablated, and single-world controls"))
display(geometry_controls[geometry_controls["aggregation"].eq("mean_across_seeds")])

observers = compact_table("T11_curvature_and_spectrum_observers.csv")
display(Markdown("## Detached curvature and Laplacian-spectrum observers"))
display(observers[observers["aggregation"].eq("mean_across_seeds")])
display(pd.DataFrame([json.loads((root / "spectral_observer_numerical_audit.json").read_text())]))

display(Markdown("## Complete compact study tables"))
for relative_name in table_manifest["display_order"]:
    frame = pd.read_csv(root / relative_name)
    print(f"\n{relative_name}: rows={len(frame)} columns={len(frame.columns)}")
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 160):
        display(frame)


In [ ]:
RAW_CSV_FILES = (
    "native_stability_invariants.csv", "resource_disclosure.csv", "training_curves.csv",
    "cohort_metrics.csv", "checkpoint_selection_interventions.csv", "causal_interventions.csv",
    "geometry_observers.csv", "paired_statistics.csv", "paired_predictions.csv",
    "claim_matrix.csv", "gate_results.csv", "failed_rows.csv", "open_scientific_rows.csv",
)
RAW_JSON_FILES = (
    "baseline_qualification_manifest.json", "qualification_manifest_audit.json",
    "fresh_seed_audit.json", "final_execution_lock_audit.json", "shared_input_audit.json",
    "label_contract_audit.json", "cross_seed_data_audit.json",
    "raw_input_internalization_audit.json", "runtime_invariance_audit.json",
    "native_training_exposure_audit.json", "native_stability_invariants.json",
    "architecture_audit.json", "parameter_audit.json", "checkpoint_manifest.json",
    "locked_native_core_audit.json", "spectral_observer_numerical_audit.json",
)
display(Markdown("## Complete machine-readable evidence"))
for filename in RAW_CSV_FILES:
    path = root / filename
    if path.exists():
        frame = pd.read_csv(path)
        print(f"\n{filename}: rows={len(frame)} columns={len(frame.columns)}")
        with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 160):
            display(frame)
for filename in RAW_JSON_FILES:
    path = root / filename
    if path.exists():
        print(f"\n{filename}")
        print(path.read_text(encoding="utf-8"))
